# Test Suite Builder — Studium KG Pipeline

This notebook:
1. **Scores** every document on richness: **length + origin info + name variants**
2. **Clusters** documents by **length × century of activity**
3. **Exports** `test_suite.jsonl` (drop-in for genai_v3)
4. **Exports** a small repeat-test set for LLM variance measurement

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
INPUT_FILE   = "/kaggle/input/studium-llm-ready-people/studium_llm_ready_people.jsonl"
OUTPUT_FILE  = "test_suite.jsonl"        # final test set for the pipeline
PROFILE_FILE = "corpus_profiles.jsonl"  # full scored corpus (for exploration)

N_PER_CLUSTER = 12    # documents sampled per (length x century) cluster
TOP_N_RICHEST = 10    # top-N always included regardless of cluster
REPEAT_N      = 5     # documents in the repeatability test set

# Length bucket thresholds (chars) — tune after seeing the distribution plot
SHORT_MAX  = 600
MEDIUM_MAX = 1400
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import json, re, collections, random
from pathlib import Path

try:
    import pandas as pd
    import matplotlib.pyplot as plt
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False
    print("pandas/matplotlib not available — text summaries only")

print("Imports OK")

## 1 — Scoring

**Richness = length + name variants + origin presence**

Other features (grades, universities) are stored as informational columns but do **not** affect the score.

In [ ]:
GRADE_KW = ["bachelier", "licencié", "maître", "docteur", "magister",
            "baccalarius", "licenciatus", "doctor"]
UNI_KW   = ["paris", "padoue", "bologne", "ferrare", "louvain",
            "oxford", "montpellier", "cologne", "avignon", "prague"]


def length_bucket(tlen: int) -> str:
    if tlen < SHORT_MAX:   return "short"
    if tlen < MEDIUM_MAX:  return "medium"
    return "long"


def century_bucket(year) -> str:
    """1435 → '15th_century', None → 'unknown'"""
    if year is None:
        return "unknown"
    suffixes = {1: "st", 2: "nd", 3: "rd"}
    c = (int(year) - 1) // 100 + 1
    sfx = suffixes.get(c if c < 4 else 0, "th")
    return f"{c}{sfx}_century"


def compute_profile(record: dict) -> dict:
    meta = record.get("meta_entities", {})
    text = record.get("text", "").lower()
    tlen = record.get("text_len", len(text))

    # ── Richness signals (your 3 criteria) ───────────────────────────────
    n_names = len(meta.get("names", []))

    has_origin = int(
        bool(meta.get("places"))
        or "birthplace" in text or "naissance" in text
        or "pays-bas" in text or "normandie" in text
    )
    has_diocese = int(
        "diocèse" in text or "diocese" in text
        or any("diocèse" in i.lower() or "diocese" in i.lower()
               for i in meta.get("institutions", []))
    )

    richness_score = (
        tlen            * 0.04
        + n_names       * 8
        + has_origin    * 10
        + has_diocese   * 8
        + (has_origin and has_diocese) * 5  # bonus when both present
    )

    # ── Informational features (not in score) ────────────────────────────
    grade_count = sum(text.count(kw) for kw in GRADE_KW)
    n_univ      = sum(1 for u in UNI_KW if u in text)
    n_places    = len(meta.get("places", []))
    n_dates     = len(meta.get("dates",  []))
    n_inst      = len(meta.get("institutions", []))

    # ── Activity year → century ──────────────────────────────────────────
    act_year = record.get("activityMediane")
    if act_year is None:
        m = re.search(r"activityMediane[:\s]+(\d{4})", record.get("text", ""))
        if m:
            act_year = int(m.group(1))

    century = century_bucket(act_year)

    return {
        "reference":      str(record.get("reference", "")),
        "name":           record.get("name", ""),
        "text_len":       tlen,
        "activity_year":  act_year,
        "century":        century,
        "n_names":        n_names,
        "has_origin":     has_origin,
        "has_diocese":    has_diocese,
        "grade_count":    grade_count,
        "n_universities": n_univ,
        "n_places":       n_places,
        "n_dates":        n_dates,
        "n_institutions": n_inst,
        "richness_score": round(richness_score, 1),
        "cluster":        f"{length_bucket(tlen)}__{century}",
    }

In [ ]:
profiles    = []
raw_records = {}   # reference → full original record (needed for export)

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rec     = json.loads(line)
        profile = compute_profile(rec)
        profiles.append(profile)
        raw_records[profile["reference"]] = rec

print(f"Loaded {len(profiles):,} documents")
with open(PROFILE_FILE, "w", encoding="utf-8") as f:
    for p in profiles:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")
print(f"Profiles saved → {PROFILE_FILE}")

## 2 — Explore distributions

In [ ]:
if HAS_PANDAS:
    df = pd.DataFrame(profiles)

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle("Corpus profile overview", fontsize=13, y=1.01)

    # Richness distribution
    axes[0,0].hist(df["richness_score"], bins=60, color="#5b6abf", edgecolor="white")
    axes[0,0].set_title("Richness score")
    axes[0,0].set_xlabel("score"); axes[0,0].set_ylabel("# docs")

    # Text length
    axes[0,1].hist(df["text_len"], bins=60, color="#3a9e8e", edgecolor="white")
    axes[0,1].axvline(SHORT_MAX,  color="#e05c3a", lw=1.5, ls="--", label=f"short < {SHORT_MAX}")
    axes[0,1].axvline(MEDIUM_MAX, color="#c97d2a", lw=1.5, ls="--", label=f"medium < {MEDIUM_MAX}")
    axes[0,1].set_title("Text length (chars)")
    axes[0,1].set_xlabel("chars"); axes[0,1].legend(fontsize=9)

    # Century distribution
    df["century"].value_counts().sort_index().plot(
        kind="bar", ax=axes[1,0], color="#c97d2a", edgecolor="white")
    axes[1,0].set_title("Activity century")
    axes[1,0].tick_params(axis="x", rotation=30)

    # Cluster heatmap (length × century)
    ax = axes[1,1]
    pivot = (
        df.assign(length=df["text_len"].apply(length_bucket))
          .groupby(["century", "length"]).size()
          .unstack(fill_value=0)
    )
    for col in ["short", "medium", "long"]:
        if col not in pivot.columns: pivot[col] = 0
    pivot = pivot[["short","medium","long"]].sort_index()
    im = ax.imshow(pivot.values, aspect="auto", cmap="Blues")
    ax.set_xticks(range(3)); ax.set_xticklabels(["short","medium","long"])
    ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index, fontsize=9)
    ax.set_title("Cluster heatmap (length × century)")
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            v = pivot.values[i,j]
            ax.text(j, i, str(v), ha="center", va="center", fontsize=9,
                    color="white" if v > pivot.values.max()*0.5 else "black")
    plt.colorbar(im, ax=ax, shrink=0.8)

    plt.tight_layout()
    plt.savefig("corpus_overview.png", dpi=120, bbox_inches="tight")
    plt.show()

    print("\nRichness score statistics:")
    print(df["richness_score"].describe().round(1).to_string())
    print("\nDocs per century:", df["century"].value_counts().sort_index().to_dict())

else:
    scores = [p["richness_score"] for p in profiles]
    n = len(scores)
    print(f"Richness — min={min(scores):.1f}  max={max(scores):.1f}  mean={sum(scores)/n:.1f}")
    century_dist = collections.Counter(p["century"] for p in profiles)
    for c, cnt in sorted(century_dist.items()):
        print(f"  {c}: {cnt}")

In [ ]:
# Top-20 richest — inspect before selecting your test set
top20 = sorted(profiles, key=lambda x: x["richness_score"], reverse=True)[:20]

if HAS_PANDAS:
    cols = ["reference","name","richness_score","text_len",
            "n_names","has_origin","has_diocese","cluster"]
    print(pd.DataFrame(top20)[cols].to_string(index=False))
else:
    for p in top20:
        print(f"ref={p['reference']:<8} score={p['richness_score']:<6} "
              f"len={p['text_len']:<5} names={p['n_names']} "
              f"origin={p['has_origin']} diocese={p['has_diocese']} "
              f"cluster={p['cluster']}  {p['name']}")

## 3 — Build the test set

**Two-tier strategy:**
- **Tier 1** — top `TOP_N_RICHEST` documents: the hardest possible inputs for the LLM
- **Tier 2** — `N_PER_CLUSTER` per (length × century) cluster, chosen by descending richness within each cluster

This gives broad temporal and length coverage while still prioritising rich documents.

In [ ]:
random.seed(42)

selected_refs = set()
test_meta     = []

# ── Tier 1: top richest ───────────────────────────────────────────────────────
for p in sorted(profiles, key=lambda x: x["richness_score"], reverse=True)[:TOP_N_RICHEST]:
    selected_refs.add(p["reference"])
    test_meta.append({**p, "selection_reason": "top_richest"})

print(f"Tier 1 (top richest): {len(selected_refs)} documents")

# ── Tier 2: stratified per cluster ───────────────────────────────────────────
by_cluster = collections.defaultdict(list)
for p in profiles:
    by_cluster[p["cluster"]].append(p)

tier2 = 0
skipped = []
for cluster, members in sorted(by_cluster.items()):
    if len(members) < 3:
        skipped.append((cluster, len(members)))
        continue
    added = 0
    for p in sorted(members, key=lambda x: x["richness_score"], reverse=True):
        if added >= N_PER_CLUSTER:
            break
        if p["reference"] not in selected_refs:
            selected_refs.add(p["reference"])
            test_meta.append({**p, "selection_reason": f"cluster:{cluster}"})
            added += 1
            tier2 += 1

print(f"Tier 2 (stratified):  {tier2} documents")
print(f"Total test set:       {len(selected_refs)} documents")
if skipped:
    print(f"Skipped {len(skipped)} tiny clusters (<3 docs): "
          + ", ".join(f"{c}({n})" for c,n in skipped))

## 4 — Export `test_suite.jsonl`

In [ ]:
exported = 0
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for meta in test_meta:
        ref = meta["reference"]
        if ref not in raw_records:
            print(f"  WARNING: reference {ref} not found — skipped")
            continue
        rec = dict(raw_records[ref])
        rec["_test_cluster"]          = meta["cluster"]
        rec["_test_richness_score"]   = meta["richness_score"]
        rec["_test_selection_reason"] = meta["selection_reason"]
        rec["_test_century"]          = meta["century"]
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        exported += 1

print(f"Exported {exported} documents → {OUTPUT_FILE}")

## 5 — Test set summary

In [ ]:
if HAS_PANDAS:
    tdf = pd.DataFrame(test_meta)

    print("=" * 55)
    print(f"  Total             : {len(tdf)}")
    print(f"  Richness range    : {tdf['richness_score'].min():.1f} – {tdf['richness_score'].max():.1f}")
    print(f"  Avg text length   : {tdf['text_len'].mean():.0f} chars")
    print(f"  With origin       : {tdf['has_origin'].sum()} ({tdf['has_origin'].mean()*100:.0f}%)")
    print(f"  With diocese      : {tdf['has_diocese'].sum()} ({tdf['has_diocese'].mean()*100:.0f}%)")
    print(f"  Avg name variants : {tdf['n_names'].mean():.1f}")
    print("=" * 55)

    print("\nDocs per cluster:")
    cs = (
        tdf.groupby("cluster")
           .agg(n=("reference","count"),
                avg_score=("richness_score","mean"),
                avg_len=("text_len","mean"),
                origin_pct=("has_origin","mean"))
           .sort_index()
    )
    cs["avg_score"]  = cs["avg_score"].round(1)
    cs["avg_len"]    = cs["avg_len"].round(0).astype(int)
    cs["origin_pct"] = (cs["origin_pct"]*100).round(0).astype(int)
    print(cs.to_string())

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle("Test set profile", fontsize=13)
    tdf["richness_score"].hist(bins=25, ax=axes[0], color="#5b6abf", edgecolor="white")
    axes[0].set_title("Richness score"); axes[0].set_xlabel("score")
    tdf["text_len"].hist(bins=25, ax=axes[1], color="#3a9e8e", edgecolor="white")
    axes[1].set_title("Text length")
    tdf["century"].value_counts().sort_index().plot(
        kind="bar", ax=axes[2], color="#c97d2a", edgecolor="white")
    axes[2].set_title("Century distribution")
    axes[2].tick_params(axis="x", rotation=30)
    plt.tight_layout()
    plt.savefig("test_suite_summary.png", dpi=120, bbox_inches="tight")
    plt.show()

else:
    print(f"Total: {len(test_meta)}")
    for c, n in sorted(collections.Counter(m["cluster"] for m in test_meta).items()):
        print(f"  {n:>3}  {c}")

## 6 — Repeatability test set

Exports the `REPEAT_N` richest documents from the highest-scoring cluster.
Run `genai_v3` on this file **3 times** and compare outputs to measure variance.

In [ ]:
# Find the cluster with the highest average richness that has enough members
best_cluster, best_members = max(
    [(c, m) for c, m in by_cluster.items() if len(m) >= REPEAT_N],
    key=lambda kv: sum(p["richness_score"] for p in kv[1]) / len(kv[1])
)

repeat_docs = sorted(best_members, key=lambda x: x["richness_score"], reverse=True)[:REPEAT_N]

repeat_file = f"repeat_test__{best_cluster.replace('__','_')}.jsonl"
with open(repeat_file, "w", encoding="utf-8") as f:
    for p in repeat_docs:
        ref = p["reference"]
        if ref in raw_records:
            rec = dict(raw_records[ref])
            rec["_repeat_cluster"]        = best_cluster
            rec["_repeat_richness_score"] = p["richness_score"]
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"Best cluster : '{best_cluster}'")
print(f"Exported {len(repeat_docs)} documents → {repeat_file}")
print()
for p in repeat_docs:
    print(f"  ref={p['reference']:<8} score={p['richness_score']:<6} "
          f"len={p['text_len']:<5} names={p['n_names']} "
          f"origin={p['has_origin']} diocese={p['has_diocese']}  "
          f"{p['name']}")

## 7 — Inspect a document

Change `INSPECT_IDX` to browse documents from the test set before running the pipeline.

In [ ]:
INSPECT_IDX = 0   # 0 = richest document in test set

ranked = sorted(test_meta, key=lambda x: x["richness_score"], reverse=True)
target = ranked[INSPECT_IDX]
rec    = raw_records[target["reference"]]

print(f"Reference      : {target['reference']}")
print(f"Name           : {target['name']}")
print(f"Richness score : {target['richness_score']}")
print(f"Cluster        : {target['cluster']}")
print(f"Text length    : {target['text_len']} chars")
print(f"Name variants  : {target['n_names']}")
print(f"Has origin     : {bool(target['has_origin'])}  "
      f"| Has diocese  : {bool(target['has_diocese'])}")
print(f"Grade count    : {target['grade_count']}  "
      f"| Universities : {target['n_universities']}")
print("\n" + "─" * 60)
print(rec.get("text", "(no text field)"))